In [ ]:
import sys
from pathlib import Path

import yaml

sys.path.insert(0, str(Path().absolute().parent.parent.parent))

In [ ]:
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns
# sns.set_theme(rc={'text.usetex' : True})
import numpy as np
import pandas as pd
from test.lib.files_utils import SimulationOutput, LisLoader, ISOTOPES, func

In [ ]:
lis_loader = LisLoader(Path().absolute().parent / 'data' / 'LIS_Default2020_Proton')

In [ ]:
output_files = Path().absolute().parent / 'data' / 'uncertainty' / 'outputs'
rows = []
for of in sorted(output_files.rglob('*.yaml')):
    date, _, part = of.parent.name.split('_')
    print(date, part)
    outputs = SimulationOutput.from_yaml(yaml.load(of.read_text(), Loader=yaml.CLoader))
    fluxes = outputs.modulate(lis_loader)
    for i, (out, flux) in enumerate(zip(outputs, fluxes)):
        # print(flux.rig_flux.rigidity)
        # print(flux.rig_flux.to_energy(ISOTOPES.get('proton')).energy)
        # for r, f in zip(*flux.rig_flux):
        for j, (r, f) in enumerate(zip(*flux.rig_flux.to_energy(ISOTOPES.get('proton')))):
            nbins = np.trim_zeros(out[ISOTOPES.get('proton')].output_dist[j], 'f').shape[0]
            rows.append([date, part, i, r, f, nbins])

In [ ]:
df = pd.DataFrame(rows, columns=['date', 'part', 'param', 'rig', 'flux', 'nbins'])
df['part'] = df['part'].astype(int)
df['param'] = df['param'].astype(int)
df['rig'] = df['rig'].astype(float)
df['flux'] = df['flux'].astype(float)
df['nbins'] = df['nbins'].astype(int)
df.sort_values(df.columns.to_list(), inplace=True)
df.to_csv('uncertainty.csv', index=False)
df

In [ ]:
df2 = df.copy()
df2 = df2.groupby(['date', 'part', 'rig']).agg(nbins=('nbins', 'max'), flux_mean=('flux', 'mean'),
                                               flux_std=('flux', 'std')).reset_index()
df2['pois'] = (df2['part'].astype(float)) ** (-1 / 2)
df2['ratio'] = (df2['flux_std'] / df2['flux_mean']) / df2['pois']
df2['ratio_norm'] = df2['ratio'] / df2['nbins']

df2['nbins_norm'] = df2['nbins'] /  df2['nbins'].max()

df2

In [ ]:
def threshold(t):
    return np.sqrt(2) * (1.29 * t**(-0.85)) / (t**-0.85 + 1.5)


fig, ax = plt.subplots(figsize=(10, 9))
axx = sns.lineplot(
    df2.query('(rig<1001)'),
    x='rig',
    y='ratio',
    hue='date',
    # errorbar=('ci', .05),
    err_style='bars', marker='o', linestyle='',
    ax=ax
)
sns.lineplot(
    x=df2.query('(rig<101)')['rig'],
    y=threshold(df2.query('(rig<101)')['rig']),
    color='k',
    label='old threshold',
    ax=ax,
)

sns.lineplot(
    df2,
    x='rig',
    y='nbins_norm',
    hue='date',
)

# axx.axhline(2*10**-2)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))
axx = sns.boxplot(
    df2.query('(rig<1001)'),
    x='rig',
    y='ratio_norm',
    hue='part',
    # errorbar=('ci', .05),
    # err_style='bars', marker='o', linestyle='',
    palette='muted',
    ax=ax
)

axx.set(xlabel='Energy', ylabel=r'(sigma sim) / (sigma pois)')
plt.xticks(rotation=90)